# LTX-2.5 Pilot — Tek Sahne, h3süz (23 commit'li keyframe + Edge TTS)

Bu pilotta **H3/MiniMax resim üreteci KULLANILMIYOR.** Video, repoya commit'li keyframe'lerden (ilk `001_street` → son `002_coffin`) LTX-2.5 **FLF2V** ile üretilir. Yeni resim üretilmez.

**Ön koşullar (bir kez, elle):**
1. Runtime → Change runtime type → **GPU (L4 24GB)**
2. Sol menü → Secrets → **HF_TOKEN** ekle (Hugging Face token'ın). *Secrets yalnızca Colab UI'dan okunur; headless/agent kernel'den okunamaz.*
3. https://huggingface.co/Lightricks/LTX-2.5 adresinde **lisansı kabul et** (gated repo; onay gelmeden model inmez).

**Ses:** Chatterbox pasif (torch çakışması) → **Edge TTS**, `adaptation.yaml` narrator'uyle (`en-US-BrianMultilingualNeural`, sahne 1 faithful). TTS alt süreçte koşulur; TTS başarısız olursa **tam süre sessiz mp3** üretilir, video yine render edilir.

Hücreleri 1 → 4 sırayla çalıştır (Run all). Hücre 4 ~15-25 dk sürer (36 GB model indirme + kurulum + 14 sn render).


In [ ]:
#@title 1. Repoyu çek ve kur (idempotent) { display-mode: "form" }
REPO_URL = "https://github.com/Yuki-TR/mobydick.git"  #@param {type:"string"}
REQUIRED_COMMIT = "84ae547f222dd7f75cc512b8b01d4a46e0fd2038"  #@param {type:"string"}
import pathlib, subprocess
ROOT = pathlib.Path("/content/mobydick")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", REQUIRED_COMMIT or "origin/main"], cwd=ROOT, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
print("checkout:", head)
if REQUIRED_COMMIT and head != REQUIRED_COMMIT:
    raise RuntimeError(f"Beklenen commit {REQUIRED_COMMIT}, gelen {head}")
subprocess.run(["pip", "install", "-q", "-e", str(ROOT)], check=True)
subprocess.run(["pip", "install", "-q", "edge-tts"], check=True)
print("kurulum tamam")


In [ ]:
#@title 2. Pinli LTX-2.5 manifest'i üret (SHA'lar HF'den) { display-mode: "form" }
from google.colab import userdata
import os, subprocess
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
subprocess.run(["python", "scripts/ltx25/fetch_manifest.py", "--output", "manifest.ltx25.json"], cwd=ROOT, check=True)
print(open(ROOT / "manifest.ltx25.json").read()[:600])


In [ ]:
#@title 3. Ses: Edge TTS anlatım (alt süreçte; fallback: tam süre sessiz mp3) { display-mode: "form" }
DURATION = 14  #@param {type:"integer"}
import subprocess, pathlib
import imageio_ffmpeg
ROOT = pathlib.Path("/content/mobydick")
raw = ROOT / "pilot_narration_raw.mp3"
padded = ROOT / "pilot_narration.mp3"
text = ("Call me Ishmael. I thought I would sail about a little and see the "
        "watery part of the world.")
voice = "en-US-BrianMultilingualNeural"
ff = imageio_ffmpeg.get_ffmpeg_exe()
tts_ok = False
try:
    r = subprocess.run(["edge-tts", "--text", text, "--voice", voice,
                        "--write-media", str(raw)], capture_output=True, text=True, cwd=ROOT)
    tts_ok = raw.is_file() and raw.stat().st_size > 1000
    print("Edge TTS rc:", r.returncode, "| raw:", (raw.stat().st_size if raw.is_file() else 0), "bytes")
    if r.returncode != 0:
        print(r.stderr[:300])
except Exception as e:
    print("Edge TTS hata:", type(e).__name__, str(e)[:200])
if tts_ok:
    subprocess.run([ff, "-y", "-i", str(raw), "-af", f"apad=whole_dur={DURATION}",
                    "-c:a", "libmp3lame", "-b:a", "192k", str(padded)], check=True)
else:
    subprocess.run([ff, "-y", "-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo",
                    "-t", str(DURATION), "-c:a", "libmp3lame", "-b:a", "192k", str(padded)], check=True)
print("narration hazır:", padded, padded.stat().st_size, "bytes (tts_ok=%s)" % tts_ok)


In [ ]:
#@title 4. Pilotu çalıştır: kurulum+model indirme → ComfyUI → FLF2V render → mux (~15-25 dk) { display-mode: "form" }
DURATION = 14  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
import subprocess, pathlib
ROOT = pathlib.Path("/content/mobydick")
narration = str(ROOT / "pilot_narration.mp3")
cmd = [
    "python", "scripts/ltx25/pilot_run.py",
    "--manifest", "manifest.ltx25.json",
    "--first", "projects/moby_dick_pilot/assets/keyframes/001_street.png",
    "--last", "projects/moby_dick_pilot/assets/keyframes/002_coffin.png",
    "--narration-file", narration,
    "--duration", str(DURATION),
]
if SEED:
    cmd += ["--seed", str(SEED)]
proc = subprocess.run(cmd, cwd=ROOT)
raise SystemExit(proc.returncode)
